# Введение в обработку текста на естественном языке

Материалы:
* Макрушин С.В. Лекция 9: Введение в обработку текста на естественном языке\
* https://realpython.com/nltk-nlp-python/
* https://scikit-learn.org/stable/modules/feature_extraction.html

In [1]:
import pandas as pd
from nltk import word_tokenize, sent_tokenize
import numpy as np
from nltk.metrics.distance import edit_distance
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer
from collections import Counter
from nltk.corpus import stopwords

## Разминка

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
import pymorphy2
from nltk.metrics.distance import edit_distance

1. Считайте слова из файла `litw-win.txt` и запишите их в список `words`. В заданном предложении исправьте все опечатки, заменив слова с опечатками на ближайшие (в смысле расстояния Левенштейна) к ним слова из списка `words`. Считайте, что в слове есть опечатка, если данное слово не содержится в списке `words`. 

In [3]:
text = '''с велечайшим усилием выбравшись из потока убегающих людей Кутузов со свитой уменьшевшейся вдвое поехал на звуки выстрелов русских орудий'''

In [4]:
fname = r"E:\Downloads\TOBD_2021_datasets\food_com\examples\litw-win.txt"
words = open(fname, 'r', encoding='windows-1251').readlines()
words = [word.strip().split()[-1] for word in words]

In [9]:
min(words, key=lambda k: edit_distance('велечайшим', k))

'величайшим'

In [8]:
find_closest('велечайшим', 1)

['величайшим']

2. Разбейте текст из формулировки задания 1 на слова; проведите стемминг и лемматизацию слов.

In [37]:
text = '''Считайте слова из файла `litw-win.txt` и запишите их в список `words`. В заданном предложении исправьте все опечатки, заменив слова с опечатками на ближайшие (в смысле расстояния Левенштейна) к ним слова из списка `words`. Считайте, что в слове есть опечатка, если данное слово не содержится в списке `words`. '''
words = word_tokenize(text)


In [40]:
stemmer = SnowballStemmer('russian')
[stemmer.stem(w) for w in words][:5]

['счита', 'слов', 'из', 'файл', '`']

In [43]:
morph = pymorphy2.MorphAnalyzer()
[morph.parse(w)[0].normalized.word for w in words][:5]

['считать', 'слово', 'из', 'файл', '`']

3. Преобразуйте предложения из формулировки задания 1 в векторы при помощи `CountVectorizer`.

In [30]:
text = '''Считайте слова из файла `litw-win.txt` и запишите их в список `words`. В заданном предложении исправьте все опечатки, заменив слова с опечатками на ближайшие (в смысле расстояния Левенштейна) к ним слова из списка `words`. Считайте, что в слове есть опечатка, если данное слово не содержится в списке `words`. '''
sents = sent_tokenize(text)
cv = CountVectorizer()
sents_cv = cv.fit_transform(sents)

In [36]:
cv.vocabulary_

{'считайте': 32,
 'слова': 24,
 'из': 12,
 'файла': 33,
 'litw': 0,
 'win': 2,
 'txt': 1,
 'запишите': 11,
 'их': 14,
 'список': 31,
 'words': 3,
 'заданном': 9,
 'предложении': 22,
 'исправьте': 13,
 'все': 5,
 'опечатки': 21,
 'заменив': 10,
 'опечатками': 20,
 'на': 16,
 'ближайшие': 4,
 'смысле': 27,
 'расстояния': 23,
 'левенштейна': 15,
 'ним': 18,
 'списка': 29,
 'что': 34,
 'слове': 25,
 'есть': 8,
 'опечатка': 19,
 'если': 7,
 'данное': 6,
 'слово': 26,
 'не': 17,
 'содержится': 28,
 'списке': 30}

In [33]:
sents_cv.toarray()

array([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
       [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
        1, 1, 2, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1]], dtype=int64)

## Лабораторная работа 9

### Расстояние редактирования

1.1 Загрузите предобработанные описания рецептов из файла `preprocessed_descriptions.csv`. Получите набор уникальных слов `words`, содержащихся в текстах описаний рецептов (воспользуйтесь `word_tokenize` из `nltk`). 

In [2]:
descriptions = pd.read_csv('E:/Downloads/preprocessed_descriptions.csv')

In [3]:
words_set = set()
words_set.update(w for d in descriptions['preprocessed_descriptions'].str.lower().dropna() 
                   for w in word_tokenize(d))

In [4]:
words = list(words_set)

1.2 Сгенерируйте 5 пар случайно выбранных слов и посчитайте между ними расстояние редактирования.

In [5]:
a = np.random.choice(words, 5)
b = np.random.choice(words, 5)
for x, y in zip(a, b):
    print('{}, {} -> {}'.format(x, y, edit_distance(x, y)))

pugliese, barbari -> 8
cobb, anne -> 4
hydration, joumou -> 8
serrano, stoup -> 6
cartoons, lenora -> 7


1.3 Напишите функцию, которая для заданного слова `word` возвращает `k` ближайших к нему слов из списка `words` (близость слов измеряется с помощью расстояния Левенштейна)

In [6]:
def find_closest(word, k):
    return sorted(words, key=lambda x: edit_distance(word, x))[:k]

In [7]:
find_closest('califonia', 5)

['california', 'californian', 'californians', 'catalonia', 'calorie']

### Стемминг, лемматизация

2.1 На основе результатов 1.1 создайте `pd.DataFrame` со столбцами: 
    * word
    * stemmed_word 
    * normalized_word 

Столбец `word` укажите в качестве индекса. 

Для стемминга воспользуйтесь `SnowballStemmer`, для нормализации слов - `WordNetLemmatizer`. Сравните результаты стемминга и лемматизации.

In [8]:
snb_stemmer_ru = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

stemmed_words = [snb_stemmer_ru.stem(word) for word in words]
normalized_words = [lemmatizer.lemmatize(word) for word in words]

words_df = pd.DataFrame()
words_df['word'] = words
words_df['stemmed_words'] = stemmed_words
words_df['normalized_words'] = normalized_words
words_df.sample(20)

,word,stemmed_words,normalized_words
602,tortilla,tortilla,tortilla
14318,babes,babe,babe
20442,date,date,date
10800,swirled,swirl,swirled
20807,nondescript,nondescript,nondescript
22093,hazlenut,hazlenut,hazlenut
24010,trinidad,trinidad,trinidad
5038,madly,mad,madly
1284,glas,glas,glas
22762,hello,hello,hello


2.2. Удалите стоп-слова из описаний рецептов. Какую долю об общего количества слов составляли стоп-слова? Сравните топ-10 самых часто употребляемых слов до и после удаления стоп-слов.

In [9]:
words = list()
words.extend(w for d in descriptions['preprocessed_descriptions'].str.lower().dropna() for w in word_tokenize(d))

In [10]:
print(Counter(words).most_common(10))

[('the', 40413), ('a', 35131), ('and', 30585), ('i', 27945), ('this', 27181), ('to', 23598), ('it', 23300), ('is', 20306), ('of', 18405), ('for', 16023)]


In [11]:
en_stopwords = stopwords.words('english')
words_wo_stop = [word for word in words if word not in en_stopwords]
print(Counter(words_wo_stop).most_common(10))

[('recipe', 15198), ('make', 6438), ('time', 5287), ('use', 4652), ('great', 4522), ('like', 4276), ('easy', 4263), ('one', 4018), ('good', 3887), ('made', 3874)]


### Векторное представление текста

3.1 Выберите случайным образом 5 рецептов из набора данных. Представьте описание каждого рецепта в виде числового вектора при помощи `TfidfVectorizer`

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cosine
from itertools import product
vectorizer = TfidfVectorizer()

In [13]:
samples = descriptions.sample(5)
samples

,name,preprocessed_descriptions
24138,shrimp stuffed green peppers,bell peppers stuffed with a tasty mixture of r...
17602,microwave shrimp in garlic butter,when i can catch it on sale i love to pick up...
20393,pecan spice cake,i have not made this yet but it is here for...
26604,sweet and sour crock pot seitan,a very flavorful simple to throw together mea...
25081,spaghetti with lemon chickpeas and bacon,recipe from the denver post yesterday sounds...


3.2 Вычислите близость между каждой парой рецептов, используя косинусное расстояние (`scipy.spatial.distance.cosine`) Результаты оформите в виде таблицы `pd.DataFrame`. В качестве названий строк и столбцов используйте названия рецептов.

In [21]:
vectors = vectorizer.fit_transform(samples['preprocessed_descriptions']).toarray()
N = len(samples)
sim = np.zeros((N, N))
for i, j in product(range(N), range(N)):
    sim[i, j] = cosine(vectors[i], vectors[j])
info = pd.DataFrame(data=sim, index=samples['name'], columns=samples['name']) 
info

name,shrimp stuffed green peppers,microwave shrimp in garlic butter,pecan spice cake,sweet and sour crock pot seitan,spaghetti with lemon chickpeas and bacon
name,,,,,
shrimp stuffed green peppers,0.000000,0.918215,0.880982,0.839200,0.809861
microwave shrimp in garlic butter,0.918215,0.000000,0.749611,0.792127,0.977487
pecan spice cake,0.880982,0.749611,0.000000,0.807845,0.960322
sweet and sour crock pot seitan,0.839200,0.792127,0.807845,0.000000,0.866930
spaghetti with lemon chickpeas and bacon,0.809861,0.977487,0.960322,0.866930,0.000000


3.3 Какие рецепты являются наиболее похожими? Прокомментируйте результат